# Ingesta

In [14]:
import shutil
import logging 
import os

# Crear carpeta logs en la raíz del proyecto
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename='logs/ingesta.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Rutas correctas desde notebooks/
origen  = "../../Telco.csv"
destino = "../data/raw/Telco.csv"

try:
    os.makedirs("../data/raw", exist_ok=True)
    shutil.copy(origen, destino)
    logging.info("Archivo copiado correctamente")
    print("Ingesta completada")
except Exception as e:
    logging.error(f"Error en ingesta: {e}")
    print(f"Error: {e}")

Ingesta completada


In [2]:
%pip install pandas
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import pandas as pd

df = pd.read_csv("../data/raw/Telco.csv")
df2 = pd.read_csv("../data/raw/Telco.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# 🔍 Paso 2: Exploración inicial

Analiza el dataset y responde:

- ¿Hay valores nulos?
- ¿Existen duplicados?
- ¿Hay datos inconsistentes?
- ¿Hay errores en tipos de datos?

Ejecuta las siguientes celdas para explorar.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

# 🧹 Paso 3: Limpieza de datos

Los titulos estan escritos con camellCase, en los valores hay mezclas con caracteres especiales (como "Fiber optic", "Month-to-month", o "Bank transfer (automatic)"), asi que haremos los siguientes cambios:

- Los titulos cambiaran a snake_case, tambien los valores en la base de datos (por ejemplo Month-to-month a month_to_month)
- Como la mayoria es str, hay espacios que en vez de null pueden tener " ", cambiaremos los valores a numeros ya que apenas llegan a 3 opciones (no, yes, no internet service a 0,1,2)
- Limpiaremos los espacios al inicio/final


In [19]:
from sklearn.preprocessing import StandardScaler
# 1. Renombrar columnas a snake_case
column_mapping = {
    'gender': 'gender',
    'SeniorCitizen': 'senior_citizen',
    'Partner': 'partner',
    'Dependents': 'dependents',
    'tenure': 'tenure',
    'PhoneService': 'phone_service',
    'MultipleLines': 'multiple_lines',
    'InternetService': 'internet_service',
    'OnlineSecurity': 'online_security',
    'OnlineBackup': 'online_backup',
    'DeviceProtection': 'device_protection',
    'TechSupport': 'tech_support',
    'StreamingTV': 'streaming_tv',
    'StreamingMovies': 'streaming_movies',
    'Contract': 'contract',
    'PaperlessBilling': 'paperless_billing',
    'PaymentMethod': 'payment_method',
    'MonthlyCharges': 'monthly_charges',
    'TotalCharges': 'total_charges',
    'Churn': 'churn'
}

df = df.rename(columns=column_mapping)
df2 = df2.rename(columns=column_mapping)
print("Columnas normalizadas:", df.columns.tolist())
logging.info("P3.1:Titulos cambiados a snake_case")


# 2. Eliminar el ID para anonimizar y evitar ruido en la IA (se asigna a df)
df = df.drop(columns=['customerID', 'total_charges'])
df2 = df2.drop(columns=['customerID'])
print("ID eliminado. Columnas actuales:", df.columns.tolist())
df2['total_charges'] = pd.to_numeric(df2['total_charges'], errors='coerce').fillna(0)
logging.info("P3.1: Columna customerID eliminada para anonimización")

# 3. Escalar las columnas numéricas continuas restantes
scaler = StandardScaler()
df[['tenure', 'monthly_charges']] = scaler.fit_transform(df[['tenure', 'monthly_charges']])

Columnas normalizadas: ['customerID', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn']
ID eliminado. Columnas actuales: ['gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'churn']


In [20]:
# 1. Definir columnas que contienen únicamente estas clases
binary_and_internet_cols = [
    'partner', 'dependents', 'phone_service', 'online_security', 
    'online_backup', 'device_protection', 'tech_support', 
    'streaming_tv', 'streaming_movies', 'paperless_billing', 'churn'
]

# 2. Diccionario de mapeo específico
mapping_dict = {
    'Yes': 1,
    'No': 0,
    'No internet service': 2
}

# 3. Reemplazar y forzar tipo numérico entero
for col in binary_and_internet_cols:
    df[col] = df[col].map(mapping_dict).astype(int)
    df2[col] = df2[col].map(mapping_dict).astype(int)

print("Columnas convertidas a enteros:")
print(df[binary_and_internet_cols].dtypes)

Columnas convertidas a enteros:
partner              int64
dependents           int64
phone_service        int64
online_security      int64
online_backup        int64
device_protection    int64
tech_support         int64
streaming_tv         int64
streaming_movies     int64
paperless_billing    int64
churn                int64
dtype: object


In [21]:
# 1. Definición de diccionarios de mapeo específicos
gender_map = {'Male': 1, 'Female': 0}
multiple_lines_map = {'No': 0, 'Yes': 1, 'No phone service': 2}
internet_service_map = {'No': 0, 'DSL': 1, 'Fiber optic': 2}
contract_map = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}

payment_method_map = {
    'Electronic check': 0,
    'Mailed check': 1,
    'Bank transfer (automatic)': 2,
    'Credit card (automatic)': 3
}

# 2. Aplicar los mapeos y transformar a tipo entero
df['gender'] = df['gender'].map(gender_map).astype(int)
df['multiple_lines'] = df['multiple_lines'].map(multiple_lines_map).astype(int)
df['internet_service'] = df['internet_service'].map(internet_service_map).astype(int)
df['contract'] = df['contract'].map(contract_map).astype(int)
df['payment_method'] = df['payment_method'].map(payment_method_map).astype(int)


df2['gender'] = df2['gender'].map(gender_map).astype(int)
df2['multiple_lines'] = df2['multiple_lines'].map(multiple_lines_map).astype(int)
df2['internet_service'] = df2['internet_service'].map(internet_service_map).astype(int)
df2['contract'] = df2['contract'].map(contract_map).astype(int)
df2['payment_method'] = df2['payment_method'].map(payment_method_map).astype(int)

# Verificación de tipos de datos
remaining_cols = ['gender', 'multiple_lines', 'internet_service', 'contract', 'payment_method']
print(df[remaining_cols].dtypes)
logging.info("P3.4: Mapeo de atributos completado")

gender              int64
multiple_lines      int64
internet_service    int64
contract            int64
payment_method      int64
dtype: object


In [22]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             7043 non-null   int64  
 1   senior_citizen     7043 non-null   int64  
 2   partner            7043 non-null   int64  
 3   dependents         7043 non-null   int64  
 4   tenure             7043 non-null   float64
 5   phone_service      7043 non-null   int64  
 6   multiple_lines     7043 non-null   int64  
 7   internet_service   7043 non-null   int64  
 8   online_security    7043 non-null   int64  
 9   online_backup      7043 non-null   int64  
 10  device_protection  7043 non-null   int64  
 11  tech_support       7043 non-null   int64  
 12  streaming_tv       7043 non-null   int64  
 13  streaming_movies   7043 non-null   int64  
 14  contract           7043 non-null   int64  
 15  paperless_billing  7043 non-null   int64  
 16  payment_method     7043 non-null   

In [23]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             7043 non-null   int64  
 1   senior_citizen     7043 non-null   int64  
 2   partner            7043 non-null   int64  
 3   dependents         7043 non-null   int64  
 4   tenure             7043 non-null   int64  
 5   phone_service      7043 non-null   int64  
 6   multiple_lines     7043 non-null   int64  
 7   internet_service   7043 non-null   int64  
 8   online_security    7043 non-null   int64  
 9   online_backup      7043 non-null   int64  
 10  device_protection  7043 non-null   int64  
 11  tech_support       7043 non-null   int64  
 12  streaming_tv       7043 non-null   int64  
 13  streaming_movies   7043 non-null   int64  
 14  contract           7043 non-null   int64  
 15  paperless_billing  7043 non-null   int64  
 16  payment_method     7043 non-null   

# 💾 Paso 4: Guardar dataset limpio

El dataset final será almacenado para su uso posterior.

In [24]:
df.to_csv('../data/processed/Telco_clean.csv', index=False)
df2.to_csv('../data/processed/Telco_limpio.csv', index=False)
print("Dataset limpio guardado correctamente")
logging.info("P4:Dataset limpio y anonimizado guardado correctamente")

Dataset limpio guardado correctamente
